In [0]:
from pyspark.sql.functions import current_date, date_format, current_timestamp

SCHEMA_FROM = "BRONZE"
TB_NAME_FROM = "TB_BITCOIN"
TB_FROM = f"{SCHEMA_FROM}.{TB_NAME_FROM}"

SCHEMA_DESTINY = "SILVER"
TB_NAME = "TB_BITCOIN"
TB_DESTINY = f"{SCHEMA_DESTINY}.{TB_NAME}"

In [0]:
df_tb_bitcoin_bronze = spark.read.table(f"{SCHEMA_FROM}.{TB_NAME_FROM}")

#rename column and add new columns
df_refined = (df_tb_bitcoin_bronze.selectExpr(
    "CAST(data AS DATE) as DT_DATA",
    "CAST(open AS DOUBLE) as VL_OPEN",
    "CAST(high AS DOUBLE) as VL_HIGH",
    "CAST(low AS DOUBLE) as VL_LOW",
    "CAST(close AS DOUBLE) as VL_CLOSE",
    "CAST(volume AS DOUBLE) as VL_VOLUME")
              .withColumn("DT_HR_LOAD", current_timestamp())
              .withColumn("DT_LOAD", current_date())
              .withColumn("HR_LOAD", date_format(current_timestamp(), "HH:mm:ss")))


df_refined.write.format('delta').mode('overwrite').saveAsTable(TB_DESTINY)